In [17]:
import re
import time
import yaml
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

## List of favorites

In [4]:
faves = pd.read_csv("mealime_favorites_inventory.csv")

In [8]:
faves_ids = faves["recipe_variant_id"].to_list()

In [12]:
base_url = f"https://app.mealime.com/recipe_variants/{faves_ids[0]}"

print_url = f"{base_url}/print"
nutrition_url = f"{base_url}/nutritional_info"

session = requests.Session()

recipe_response = session.get(print_url, timeout=30)
nutrition_response = session.get(nutrition_url, timeout=30)

print("Recipe:", recipe_response.status_code, len(recipe_response.content), "bytes")
print("Nutrition:", nutrition_response.status_code, len(nutrition_response.content), "bytes")

Recipe: 200 13751 bytes
Nutrition: 200 8970 bytes


In [16]:


for label, response in [
    ("Recipe", recipe_response),
    ("Nutrition", nutrition_response),
]:
    soup = BeautifulSoup(response.text, "html.parser")
    print(f"\n{label}:")
    print(soup.get_text(" ", strip=True)[:500])


Recipe:
Honey-Dijon Sausage, Cabbage & Carrot Rotini Pasta with Chives Honey-Dijon Sausage, Cabbage & Carrot Rotini Pasta with Chives 30 minutes | 4 servings Find cookware chef's knife colander cutting board garlic press (optional) grater large pot measuring cups measuring spoons mixing bowls nonstick skillet stirring spoon vegetable peeler whisk or fork Grab ingredients 4 medium carrots 1 small pkg chives 4 cloves garlic 1 lb ground pork sausage 1 medium head napa cabbage 10 oz rotini pasta black peppe

Nutrition:
Amounts per 1 serving of meal Quick facts 695 Calories 31 g Total Fat 71 g Net Carbs 29 g Protein 14 g Sugars 7 g Fiber Good source of vitamins & minerals Selenium 127 % Vitamin K 95 % B12 94 % Folate 89 % Thiamine 84 % Niacin 81 % More nutrition facts Fat Total Fat 30.66  g Monounsaturated 12.5 g Polyunsaturated 5.63 g Omega-6 4.83 g Omega-3 0.37 g Saturated 10.12 g Trans 0.14 g Cholesterol 71 mg Carbohydrates Total Carbohydrate 77.86 g Dietary Fiber 7.16 g Net Carbs 70.7 

In [35]:
def clean(text):
    return re.sub(r"\s+", " ", text).strip()


def extract_nutrition(html):
    soup = BeautifulSoup(html, "html.parser")
    nutrition = {}

    # Calories from the Quick Facts section
    for fact in soup.select(".quick-facts .fact"):
        name_el = fact.select_one(".name")
        amount_el = fact.select_one(".amount")

        if name_el and amount_el:
            name = clean(name_el.get_text())
            amount = clean(amount_el.get_text(" "))
            nutrition[name] = amount

    # Detailed fat, carbs, protein, vitamins and minerals
    for heading in soup.find_all("h3"):
        category = clean(heading.get_text())

        if category not in ["Fat", "Carbohydrates", "Protein", "Minerals"]:
            continue

        table = heading.find_next_sibling("table")
        if not table:
            continue

        for tr in table.select("tr"):
            cells = tr.find_all("td")

            if len(cells) >= 2:
                name = clean(cells[0].get_text())
                value = clean(cells[1].get_text())
                if name and value:
                    nutrition[name] = value

    # Normalize into our preferred nutrition format
    fields = {
        "Calories": "Calories",
        "Total Fat": "Total Fat",
        "Saturated": "Saturated Fat",
        "Sodium": "Sodium",
        "Total Carbohydrate": "Total Carbohydrates",
        "Dietary Fiber": "Dietary Fiber",
        "Sugars": "Total Sugars",
        "Total Protein": "Protein",
        "Cholesterol": "Cholesterol",
    }

    lines = ["Per serving:"]

    for source, label in fields.items():
        if source in nutrition:
            lines.append(f"{label}: {nutrition[source]}")

    return "\n".join(lines)

In [61]:
import re
from fractions import Fraction
from bs4 import BeautifulSoup

FRACTIONS = {
    "½": "1/2", "¼": "1/4", "¾": "3/4",
    "⅓": "1/3", "⅔": "2/3", "⅛": "1/8",
    "⅜": "3/8", "⅝": "5/8", "⅞": "7/8",
}

def parse_amount(text):
    """Convert '1 ½', '½', '1.5', etc. to a Fraction."""
    for symbol, replacement in FRACTIONS.items():
        text = text.replace(symbol, " " + replacement)

    parts = text.strip().split()
    try:
        return sum((Fraction(x) for x in parts), Fraction(0))
    except (ValueError, ZeroDivisionError):
        return None


def format_amount(amount):
    """Format quantities without unnecessary decimals."""
    whole = amount.numerator // amount.denominator
    remainder = amount - whole

    if remainder == 0:
        return str(whole)
    if whole:
        return f"{whole} {remainder}"
    return str(remainder)


def combine_quantities(quantities):
    """
    Combine repeated ingredient quantities.

    Supports tsp/tbsp conversion and identical units.
    Leaves incompatible or unrecognized units separate.
    """
    parsed = []

    for q in quantities:
        q = q.strip()

        match = re.fullmatch(
            r"([\d\s./½¼¾⅓⅔⅛⅜⅝⅞]+)\s*"
            r"(tsp|tbsp|teaspoons?|tablespoons?)?",
            q,
            flags=re.I
        )

        if not match:
            return " + ".join(quantities)

        amount = parse_amount(match.group(1))
        unit = (match.group(2) or "").lower()

        if amount is None:
            return " + ".join(quantities)

        if unit in ("teaspoon", "teaspoons"):
            unit = "tsp"
        elif unit in ("tablespoon", "tablespoons"):
            unit = "tbsp"

        parsed.append((amount, unit))

    units = {unit for _, unit in parsed}

    if units <= {"tsp", "tbsp"}:
        total_tsp = sum(
            (amount * (3 if unit == "tbsp" else 1)
             for amount, unit in parsed),
            Fraction(0)
        )

        tablespoons = total_tsp // 3
        teaspoons = total_tsp - tablespoons * 3

        parts = []
        if tablespoons:
            parts.append(f"{format_amount(tablespoons)} tbsp")
        if teaspoons:
            parts.append(f"{format_amount(teaspoons)} tsp")

        return " + ".join(parts)

    if len(units) == 1:
        total = sum((a for a, _ in parsed), Fraction(0))
        unit = next(iter(units))
        return f"{format_amount(total)} {unit}".strip()

    return " + ".join(quantities)


def extract_recipe(html, variant_id):
    soup = BeautifulSoup(html, "html.parser")

    # 1. Name, servings and cooking time
    title_el = soup.select_one(".meal-header h1")
    description_el = soup.select_one(".description")

    if not title_el or not description_el:
        raise ValueError("Missing recipe header")

    name = title_el.get_text(" ", strip=True)
    description = description_el.get_text(" ", strip=True)

    servings_match = re.search(r"(\d+)\s+servings?", description, re.I)
    time_match = re.search(r"(\d+)\s+minutes?", description, re.I)

    if not servings_match:
        raise ValueError("Serving count missing")

    servings = int(servings_match.group(1))
    minutes = int(time_match.group(1)) if time_match else None

    # 2. Extract shopping-list ingredients
    ingredients = []

    for li in soup.select("li.line-item"):
        qty_el = li.select_one(".quantity")
        name_el = li.select_one(".ingredient")

        if not name_el:
            continue

        ingredient = name_el.get_text(" ", strip=True)
        quantity = (
            qty_el.get_text(" ", strip=True).replace("\xa0", "")
            if qty_el else ""
        )

        ingredients.append({
            "name": ingredient,
            "quantity": quantity
        })

    # 3. Extract individual cooking steps
    steps = []
    step_ingredients = []

    for li in soup.select("li.instruction"):
        number_el = li.select_one(".number")
        primary_el = li.select_one(".primary")
        secondary_el = li.select_one(".secondary pre")

        if not primary_el:
            continue

        number = (
            number_el.get_text(strip=True)
            if number_el else str(len(steps) + 1)
        )

        instruction = primary_el.get_text(" ", strip=True)

        # Preserve ingredient quantities beneath each step
        if secondary_el:
            secondary_lines = [
                line.strip()
                for line in secondary_el.get_text().splitlines()
                if line.strip()
            ]

            step_ingredients.extend(secondary_lines)

            instruction += "\n" + "\n".join(secondary_lines)

        steps.append(f"{number}. {instruction}")

    # 4. Fill in missing shopping-list quantities
    #    using ingredient references in individual steps
    warnings = []

    for item in ingredients:
        if item["quantity"]:
            continue

        ingredient_name = item["name"]

        # Mealime sometimes uses more descriptive ingredient names
        # in the shopping list than in the cooking instructions.
        aliases = {
            "coriander, ground": "coriander",
            "cumin, ground": "cumin",
            "red lentils, dried": "red lentils",
            "brown lentils, dried": "brown lentils",
            "basil, dried": "basil",
            "oregano, dried": "oregano",
            "thyme, dried": "thyme",
            "turmeric, ground": "turmeric",
            "cinnamon, ground": "cinnamon",
            "ginger root": "ginger",
            "pure maple syrup": "maple syrup",
        }
        
        ingredient_name = aliases.get(
            ingredient_name.lower(),
            ingredient_name
        )

        # Match the ingredient at the END of a step ingredient line
        pattern = re.compile(
            r"^(.+?)\s+" + re.escape(ingredient_name) + r"$",
            re.I
        )

        quantities = []

        for line in step_ingredients:
            match = pattern.match(line)
            if match:
                quantities.append(match.group(1).strip())

        if quantities:
            item["quantity"] = combine_quantities(quantities)
        else:
            warnings.append(f"No quantity found: {ingredient_name}")

    # 5. Assemble the Paprika ingredient list
    ingredient_lines = []

    for item in ingredients:
        if item["quantity"]:
            ingredient_lines.append(
                f"{item['quantity']} {item['name']}"
            )
        else:
            ingredient_lines.append(item["name"])

    if not ingredient_lines or not steps:
        raise ValueError("Missing ingredients or directions")

    return {
        "name": name,
        "servings": servings,
        "minutes": minutes,
        "ingredient_lines": ingredient_lines,
        "direction_lines": steps,
        "warnings": warnings
    }

In [50]:
import base64
import requests
from io import BytesIO
from PIL import Image

def get_photo_base64(url):
    if not url:
        return None

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    # Convert to JPEG for reliable Paprika importing
    image = Image.open(BytesIO(response.content)).convert("RGB")

    # Keep file sizes reasonable
    image.thumbnail((1200, 1200))

    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=85)

    return base64.b64encode(buffer.getvalue()).decode("ascii")

In [37]:
def build_paprika_recipe(variant_id, name, print_html, nutrition_html):
    data = extract_recipe(print_html, variant_id)
    nutrition = extract_nutrition(nutrition_html)

    if "Calories:" not in nutrition:
        raise ValueError("Nutrition data missing")

    if data["warnings"]:
        print(f"Warnings for {variant_id}: {data['warnings']}")

    recipe = {
        "name": data["name"],
        "servings": f"{data['servings']} servings",
        "source": "Mealime",
        "source_url": (
            f"https://app.mealime.com/"
            f"recipe_variants/{variant_id}/print"
        ),
        "on_favorites": True,
        "categories": ["Mealime"],
        "ingredients": "\n".join(data["ingredient_lines"]),
        "directions": "\n\n".join(data["direction_lines"]),
        "nutritional_info": nutrition,
    }

    if data["minutes"]:
        recipe["total_time"] = f"{data['minutes']} min"

    return recipe

In [58]:
recipe = build_paprika_recipe(
    1503,
    "KP",
    print_html,
    nutrition_html,
)

print("INGREDIENTS:")
print(recipe["ingredients"])

print("\nDIRECTIONS:")
print(recipe["directions"])

print("\nNUTRITION:")
print(recipe["nutritional_info"])

INGREDIENTS:
2 (6 oz) jars artichoke hearts, marinated
1 ½ lb chicken breasts, boneless skinless
2 fennel bulbs
4 cloves garlic
1 small bunch Italian (flat-leaf) parsley
10 oz penne pasta
2 red bell peppers
12 sun-dried tomatoes, oil-packed
2 tbsp balsamic vinegar
1/2 tsp black pepper
1/2 tsp crushed red pepper
3 tbsp + 1 tsp extra virgin olive oil
2 tsp oregano, dried
1 tbsp salt

DIRECTIONS:
1. Preheat oven to 400°F.

2. Wash and dry the fresh produce.
2 fennel bulbs
2 red bell peppers
1 small bunch Italian (flat-leaf) parsley

3. Trim off root end and stalks of the fennel, leaving only the white bulbs. Halve fennel bulbs lengthwise, then slice crosswise into thin half-moons. Transfer to a large baking sheet pan.

4. Trim, seed, and large dice bell peppers. Add to the baking sheet with the fennel.

5. Pat chicken dry with paper towels, large dice, and add to the baking sheet. Drizzle chicken and veggies with oil, then season with spices. Stir to combine and spread out in an even laye

In [57]:
recipe

{'name': 'Italian Pasta Salad with Roasted Chicken, Fennel, Peppers & Artichokes',
 'servings': '4 servings',
 'source': 'Mealime',
 'source_url': 'https://app.mealime.com/recipe_variants/1503/print',
 'on_favorites': True,
 'categories': ['Mealime'],
 'ingredients': '2 (6 oz) jars artichoke hearts, marinated\n1 ½ lb chicken breasts, boneless skinless\n2 fennel bulbs\n4 cloves garlic\n1 small bunch Italian (flat-leaf) parsley\n10 oz penne pasta\n2 red bell peppers\n12 sun-dried tomatoes, oil-packed\n2 tbsp balsamic vinegar\n1/2 tsp black pepper\n1/2 tsp crushed red pepper\n3 tbsp + 1 tsp extra virgin olive oil\n2 tsp oregano, dried\n1 tbsp salt',
 'directions': "1. Preheat oven to 400°F.\n\n2. Wash and dry the fresh produce.\n2 fennel bulbs\n2 red bell peppers\n1 small bunch Italian (flat-leaf) parsley\n\n3. Trim off root end and stalks of the fennel, leaving only the white bulbs. Halve fennel bulbs lengthwise, then slice crosswise into thin half-moons. Transfer to a large baking sheet

In [59]:
recipes = []
errors = []

for i, row in faves.iterrows():
    variant_id = int(row["recipe_variant_id"])
    name = row["name"]

    base_url = f"https://app.mealime.com/recipe_variants/{variant_id}"

    try:
        r_print = session.get(f"{base_url}/print", timeout=30)
        r_nutrition = session.get(f"{base_url}/nutritional_info", timeout=30)

        r_print.raise_for_status()
        r_nutrition.raise_for_status()

        recipe = build_paprika_recipe(
            variant_id, name, r_print.text, r_nutrition.text
        )

        recipes.append(recipe)
        print(f"{i+1}/{len(faves)} ✓ {name}")

    except Exception as e:
        errors.append({
            "variant_id": variant_id,
            "name": name,
            "error": str(e)
        })
        print(f"{i+1}/{len(faves)} ✗ {name}: {e}")

    time.sleep(0.75)

1/84 ✓ Honey-Dijon Sausage, Cabbage & Carrot Rotini Pasta with Chives
2/84 ✓ Kidney Bean Curry (Rajma Chawal) with Rice & Mango-Cucumber Salad
3/84 ✓ "Kung Pao" Chicken with Broccoli, Bell Pepper, Baby Bok Choy & Peanuts
4/84 ✓ Crispy Chicken Burrito (Chimichanga) with Spicy Peach & Mango Salsa
5/84 ✓ Curried Cottage Cheese Fritters with Mint Raita & Mango-Pepper Salad
Warnings for 9294: ['No quantity found: coriander, ground', 'No quantity found: cumin, ground', 'No quantity found: red lentils, dried']
6/84 ✓ Indian Spiced Lentil Soup with Sweet Potato, Spinach & Pita
7/84 ✓ Gnocchi with Broccoli & Cheddar Cheese Sauce
8/84 ✓ Creamy Cajun Orzo Pasta with Butternut Squash, Mushrooms & Bell Pepper
9/84 ✓ Saucy Cashew-Lime Chicken with Basil over Zucchini & Carrot Noodles
Warnings for 6910: ['No quantity found: basil, dried', 'No quantity found: oregano, dried', 'No quantity found: thyme, dried']
10/84 ✓ Black Bean Penne Pasta with Mixed Vegetables
11/84 ✓ Thai Ground Beef & Carrot Lettu

In [54]:
for recipe, (_, row) in zip(recipes, faves.iterrows()):
    photo_url = row["image_url"]
    #photo_url = photo_urls.get(variant_id)

    if not photo_url:
        print(f"No photo URL: {variant_id}")
        continue

    try:
        recipe["photo"] = get_photo_base64(photo_url)
        print(f"✓ Photo added: {recipe['name']}")
    except Exception as e:
        print(f"✗ Photo failed: {recipe['name']}: {e}")

✓ Photo added: Honey-Dijon Sausage, Cabbage & Carrot Rotini Pasta with Chives
✓ Photo added: Kidney Bean Curry (Rajma Chawal) with Rice & Mango-Cucumber Salad
✓ Photo added: "Kung Pao" Chicken with Broccoli, Bell Pepper, Baby Bok Choy & Peanuts
✓ Photo added: Crispy Chicken Burrito (Chimichanga) with Spicy Peach & Mango Salsa
✓ Photo added: Curried Cottage Cheese Fritters with Mint Raita & Mango-Pepper Salad
✓ Photo added: Indian Spiced Lentil Soup with Sweet Potato, Spinach & Pita
✓ Photo added: Gnocchi with Broccoli & Cheddar Cheese Sauce
✓ Photo added: Creamy Cajun Orzo Pasta with Butternut Squash, Mushrooms & Bell Pepper
✓ Photo added: Saucy Cashew-Lime Chicken with Basil over Zucchini & Carrot Noodles
✓ Photo added: Black Bean Penne Pasta with Mixed Vegetables
✓ Photo added: Thai Ground Beef & Carrot Lettuce Wraps
✓ Photo added: Spinach-Pea Soup with Bacon, Mint, Bread Crumbs & Pumpkin Seeds
✓ Photo added: Cheesy Turkey, Potato, Carrot & Celery Soup
✓ Photo added: Hawaiian Pineapp

In [60]:
# Save all successfully converted recipes
with open("mealime_paprika.yml", "w", encoding="utf-8") as f:
    yaml.safe_dump(
        recipes,
        f,
        allow_unicode=True,
        sort_keys=False,
        width=1000
    )

# Save any failed recipe IDs separately
pd.DataFrame(
    errors,
    columns=["variant_id", "name", "error"]
).to_csv("mealime_errors.csv", index=False)

print(f"Successfully converted: {len(recipes)} / {len(faves)}")

Successfully converted: 84 / 84
